[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# A Complete Data Layer


## What you will be able to do

Put a data layer together from nothing: models in a module, a schema built by an Alembic migration,
the old system's CSV exports loaded with a savepoint for every row, so that a bad row is reported and
not the end of the load, the registrar's questions answered by queries, and tests that build their
database from the same migrations. Run all of it as one script against a new database, and recognize
a database that already has the tables a migration creates, a date that arrived as text, a header
with a byte-order mark, and a load lost to one bad row.


## The idea

### The problem

The college is leaving its old registrar system, which can export nothing but CSV files: courses,
terms, sections, students, and a year's enrollments. The new data layer has to build a database the
migration tool knows about, load the exports into it, and answer the registrar's questions, and it
has to do that again next month from a newer export, and on a colleague's machine from nothing.

Every part of it has been in the guide, one notebook at a time. Together they meet problems none of
them had alone. The exports are text, so every date and number arrives as a string. They were written
by another program, so a header can carry a byte-order mark, and some rows name a student who does
not exist, repeat a row, or hold a value the table refuses. A load that stops at the first bad row
loses the good ones with it, and a load that pretends the bad ones are fine hides them.

### What a data layer is

> A **data layer** is the part of a program that owns its database: the **models**, which describe
> the tables; the **migrations**, which build and change them; the **loaders**, which bring data in
> from outside; the **queries**, which answer questions; and the **tests**, which check all of it.
> Every other part of the program goes through it, and none of them writes SQL of its own. Here it
> is a folder of small modules, `models.py`, `database.py`, `load.py` and `queries.py`, with
> Alembic's `migrations`, a `conftest.py` and a test file beside them, and `build.py`, which runs
> the lot.

### Why it works that way

- **The migrations build the schema, even the first time.** A database built with `create_all` has
  no record of which revision it matches, and the next migration has nothing to start from.
- **Text becomes Python at the edge.** The loader turns every string into the type the column wants,
  `date.fromisoformat` and `int`, before the ORM sees it, so the models never have to guess.
- **A savepoint for every row keeps one refusal from undoing the rest.** The database checks the
  row, a refusal rolls back only that row's savepoint, and the loader records the line and the reason
  and goes on.
- **Tests build their database the way production does.** The test database comes from the same
  migrations, so a migration that does not work fails a test before it fails a deployment.
- **Every function takes the session it works in.** The script, the tests and this notebook each
  choose the database the same functions run against.

### Where this shows up

Moving data from an old system into a new one is one of the commonest jobs a data layer gets, and
this is its usual shape. **Migrations with Alembic** built the migration environment this notebook
uses, **Testing a Data Layer** the rolled-back session fixture, **Joins and Aggregates** the
standing report, and **The Session** notebook the savepoints. The **Files, Paths and Formats** guide
read CSV files with the `csv` module.

### What this notebook covers

- The project: models, and one place to make an engine
- The schema, from a migration
- Reference data from the exports
- A year's enrollments, a savepoint for every row
- The registrar's questions
- Tests that build their database from the migrations
- Which piece does which job
- One script, from nothing, finished
- Four errors, from tables a migration finds already there to a load lost to one bad row

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import csv
import io

from sqlalchemy import CheckConstraint, create_engine, select
from sqlalchemy.exc import IntegrityError
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column


class Base(DeclarativeBase):
    pass


class Grade(Base):
    __tablename__ = "grades"
    __table_args__ = (CheckConstraint("points BETWEEN 0 AND 4"),)
    student: Mapped[str] = mapped_column(primary_key=True)
    points: Mapped[int]


EXPORT = "student,points\nAna,4\nBen,9\nChloe,3\nAna,2\n"

engine = create_engine("sqlite://", connect_args={"autocommit": False})
Base.metadata.create_all(engine)
with Session(engine) as session:
    for line, row in enumerate(csv.DictReader(io.StringIO(EXPORT)), start=2):
        try:
            with session.begin_nested():
                session.add(Grade(student=row["student"], points=int(row["points"])))
        except IntegrityError as error:
            print(f"line {line} refused: {error.orig}")
    session.commit()
    grades = session.scalars(select(Grade).order_by(Grade.student))
    print([(grade.student, grade.points) for grade in grades])
```

```
line 3 refused: CHECK constraint failed: points BETWEEN 0 AND 4
line 5 refused: UNIQUE constraint failed: grades.student
[('Ana', 4), ('Chloe', 3)]
```

Four rows of text, two of them bad. `int()` turned each value into a number before the ORM saw it,
every row went in inside a savepoint of its own, and the two the database refused were rolled back
alone, each reported with its line and the database's reason. The good rows were committed together
at the end. That is the enrollment loader below, on a smaller table.


## Setup

Fifteen imports, one of them installed first where it is missing, the old system's exports written
as CSV files, and the helpers that run Alembic and pytest.

- `csv` writes the exports here, and `load.py` reads them with it
- `subprocess` and `sys` run Alembic and pytest with this notebook's Python, `os` passes them their
  settings, `shlex` prints each command as a shell would read it, and `re` tidies what they print.
  Colab does not have Alembic, so there the cell installs 1.20.0 with `pip`, and `version` and
  `PackageNotFoundError`, from `importlib.metadata`, do that and print the versions
- `sqlalchemy` is the library itself; `create_engine`, `event` and `StaticPool` make the engine,
  `select` and `func` count rows, `Session` opens sessions, and `IntegrityError` is what the
  database's refusals raise
- `date` is what the `Date` columns take and return
- `Path` names the files, and `shutil` removes the scratch folder at the start and at the end

The project is the scratch folder itself. Setup writes `data/`, five CSV files made from the
college's lists: the enrollments name students by email and sections by course code and term name,
as an export from another system would, and three rows at the end are bad on purpose. It makes
`engine`, which opens `college.db` when first used, and it defines `alembic`, `edit` and
`pytest_report` as the **Migrations with Alembic** and **Testing a Data Layer** notebooks did.
Unlike every notebook since **Many to Many**, Setup builds no tables: the migration does that.

Colab has SQLAlchemy and pytest installed, and this notebook runs SQLAlchemy 2.0.54, Alembic 1.20.0
and pytest 8.4.2. Any 2.0 release runs it, though an error may be worded a little differently. To
match it exactly, run `%pip install sqlalchemy==2.0.54` in a cell of its own, restart the session,
and run this cell again.


In [1]:
import csv
import os
import re
import shlex
import shutil
import subprocess
import sys
from datetime import date
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("alembic")
except PackageNotFoundError:                                        # Colab has no Alembic: install the version this notebook runs
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore", "alembic==1.20.0"],
                   check=True)

import sqlalchemy
from sqlalchemy import create_engine, event, func, select
from sqlalchemy.exc import IntegrityError
from sqlalchemy.orm import Session
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

DATA = SCRATCH / "data"
DATA.mkdir()
CODES = [code for code, title, department, credits in COURSES]
TERM_NAMES = [name for name, starts_on in TERMS]
EMAILS = [email for name, email, program, started_on in STUDENTS]
ENROLLMENT_ROWS = [(EMAILS[student - 1], CODES[(section - 1) % 10], TERM_NAMES[(section - 1) // 10], status, grade or "")
                   for student, section, status, grade in ENROLLMENTS]
EXPORTED = {
    "courses.csv": (["code", "title", "department", "credits"], COURSES),
    "terms.csv": (["name", "starts_on"], TERMS),
    "sections.csv": (["course", "term", "capacity"],
                     [(CODES[course - 1], TERM_NAMES[term - 1], capacity) for course, term, capacity in SECTIONS]),
    "students.csv": (["name", "email", "program", "started_on"], STUDENTS),
    "enrollments.csv": (["email", "course", "term", "status", "grade"], ENROLLMENT_ROWS + [
        ("znakamura@college.edu", "STA-200", "Spring 2026", "enrolled", ""),   # nobody with this email
        ENROLLMENT_ROWS[0],                                                     # the first row, sent twice
        ("areyes@college.edu", "PSY-101", "Spring 2026", "dropped", ""),        # a status the table refuses
    ]),
}
for name, (header, rows) in EXPORTED.items():
    with open(DATA / name, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(rows)
print({name: len(rows) for name, (header, rows) in EXPORTED.items()})


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine


os.environ["NO_COLOR"] = "1"                                        # no terminal codes in what the commands print
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"                         # no compiled copy of a file rewritten within a second
sys.dont_write_bytecode = True                                      # and none from this notebook's own imports
engine = college_engine(DATABASE)                                   # connects to nothing until it is used


def alembic(*arguments):
    """Run an alembic command in the project folder, and print what it printed, less the lines every command repeats."""
    engine.dispose()                                                # the notebook's own connections close first
    done = subprocess.run(
        [sys.executable, "-m", "alembic", *arguments], cwd=SCRATCH, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    lines = done.stdout.replace(f"{SCRATCH.resolve()}{os.sep}", "").splitlines()
    if "Traceback (most recent call last):" in lines:              # the error's own line, not Python's files
        error = [line for line in lines if re.match(r"[\w.]+(Error|Exception): ", line)][-1]
        lines = lines[:lines.index("Traceback (most recent call last):")] + ["Traceback (most recent call last): ...", error]
    print("$", shlex.join(["alembic", *arguments]))
    for line in lines:
        if not any(noise in line for noise in ("Context impl", "Will assume", "setting up autogenerate plugin")):
            print("   ", line)


def edit(path, old, new):
    """Replace the one place in a file where old appears with new."""
    source = Path(path).read_text()
    assert source.count(old) == 1, f"{old!r} appears {source.count(old)} times in {path}"
    Path(path).write_text(source.replace(old, new))


def pytest_report(*arguments, folder=SCRATCH):
    """What python -m pytest prints when it runs in the folder, less what differs between computers."""
    settings = {"COLUMNS": "80", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1", "PYTHONNODEBUGRANGES": "1"}
    finished = subprocess.run([sys.executable, "-m", "pytest", "--no-header", *arguments],
                              cwd=folder, capture_output=True, text=True, env={**os.environ, **settings})
    report = finished.stdout + finished.stderr
    report = report.replace(f"{Path(folder).resolve()}/", "")          # the folder's own path
    report = re.sub(r"\S*/_pytest/", "_pytest/", report)                # the path to pytest's own files
    report = re.sub(r"0x[0-9a-f]+", "0x...", report)                    # the memory addresses of objects
    return re.sub(r" in \d+\.\d+s\b", "", report).rstrip()              # the time the run took


print("alembic", version("alembic"), "| pytest", version("pytest"))


{'courses.csv': 10, 'terms.csv': 4, 'sections.csv': 40, 'students.csv': 25, 'enrollments.csv': 231}
alembic 1.20.0 | pytest 8.4.2


## Worked examples

### The project: models, and one place to make an engine

The college's classes, as every notebook from **Relationships** on has had them, go into
`models.py`:


In [2]:
%%writefile scratch/models.py
"""The college's tables, as classes."""
from datetime import date

from sqlalchemy import CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


Writing scratch/models.py


`database.py` is the one place the project makes an engine, with the foreign keys and the
`autocommit=False` that every engine in the guide has had, so that the script and the tests get the
same one:


In [3]:
%%writefile scratch/database.py
"""The one place the project makes an engine: SQLite with foreign keys enforced."""
from sqlalchemy import create_engine, event


def make_engine(url):
    engine = create_engine(url, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True                          # the PRAGMA does nothing inside a transaction
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False

    return engine


Writing scratch/database.py


### The schema, from a migration

The migration environment, as in **Migrations with Alembic**, and a first revision autogenerated
against a database with no tables at all, which makes it the revision that creates them:


In [4]:
alembic("init", "migrations")
ini = SCRATCH / "alembic.ini"
ini.write_text(re.sub(r"^sqlalchemy\.url = .*$", "sqlalchemy.url = sqlite:///college.db", ini.read_text(), flags=re.M))
edit(SCRATCH / "migrations" / "env.py", "target_metadata = None",
     "from models import Base\n\ntarget_metadata = Base.metadata")

alembic("revision", "--autogenerate", "-m", "the college", "--rev-id", "0001")
alembic("upgrade", "head")
print(sqlalchemy.inspect(engine).get_table_names())


$ alembic init migrations
    Creating directory migrations ...  done
    Creating directory migrations/versions ...  done
    Generating migrations/script.py.mako ...  done
    Generating migrations/env.py ...  done
    Generating migrations/README ...  done
    Generating alembic.ini ...  done
    Please edit configuration/connection/logging settings in alembic.ini before proceeding.
$ alembic revision --autogenerate -m 'the college' --rev-id 0001
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'courses'
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'students'
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'terms'
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'sections'
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'enrollments'
    Generating migrations/versions/0001_the_college.py ...  done
$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade  -> 0

Autogenerate compared the models with an empty database and found all five tables missing, so
revision 0001 creates them, in the order their foreign keys need. `upgrade head` ran it on
`college.db`, which SQLite made for the purpose, and Alembic's own `alembic_version` table records
that it did. The database never saw `create_all`: its tables and the record of how it got them came
from the same place.

### Reference data from the exports

`load.py` reads the exports. Every value in a CSV file is text, so the loader turns each into what
its column wants before the ORM sees it: `int()` for credits and capacity, `date.fromisoformat()` for
dates, and a lookup by code and term for the section a row names:


In [5]:
%%writefile scratch/load.py
"""Load the old system's CSV exports into the college's database."""
import csv
from datetime import date

from sqlalchemy import select
from sqlalchemy.exc import IntegrityError

from models import Course, Enrollment, Section, Student, Term


def read_rows(path):
    """Every row of a CSV file, as a dictionary keyed by the header, which may begin with a byte-order mark."""
    with open(path, newline="", encoding="utf-8-sig") as file:
        return list(csv.DictReader(file))


def load_reference(session, folder):
    """Courses, terms, sections and students: everything an enrollment refers to."""
    for row in read_rows(folder / "courses.csv"):
        session.add(Course(code=row["code"], title=row["title"], department=row["department"], credits=int(row["credits"])))
    for row in read_rows(folder / "terms.csv"):
        session.add(Term(name=row["name"], starts_on=date.fromisoformat(row["starts_on"])))
    session.flush()
    courses = {course.code: course for course in session.scalars(select(Course))}
    terms = {term.name: term for term in session.scalars(select(Term))}
    for row in read_rows(folder / "sections.csv"):
        session.add(Section(course=courses[row["course"]], term=terms[row["term"]], capacity=int(row["capacity"])))
    for row in read_rows(folder / "students.csv"):
        session.add(Student(name=row["name"], email=row["email"], program=row["program"],
                            started_on=date.fromisoformat(row["started_on"])))
    session.commit()


def load_enrollments(session, path):
    """Load every enrollment the database accepts, each in a savepoint of its own, and return the rest's line and reason."""
    students = dict(session.execute(select(Student.email, Student.id)).all())
    sections = {(code, term): section_id for section_id, code, term in
                session.execute(select(Section.id, Course.code, Term.name).join(Section.course).join(Section.term))}
    refused = []
    for line, row in enumerate(read_rows(path), start=2):           # line 1 is the header
        try:
            with session.begin_nested():
                session.add(Enrollment(student_id=students[row["email"]], section_id=sections[row["course"], row["term"]],
                                       status=row["status"], grade=row["grade"] or None))
        except KeyError as error:
            refused.append((line, f"unknown: {error.args[0]}"))
        except IntegrityError as error:
            refused.append((line, str(error.orig)))
    session.commit()
    return refused


Writing scratch/load.py


In [6]:
sys.path.insert(0, str(SCRATCH))                                    # the project's modules, importable here
from load import load_enrollments, load_reference
from models import Course, Enrollment, Section, Student, Term

with Session(engine) as session:
    load_reference(session, DATA)
    print({cls.__tablename__: session.scalar(select(func.count()).select_from(cls))
           for cls in (Course, Term, Section, Student)})


{'courses': 10, 'terms': 4, 'sections': 40, 'students': 25}


Ten courses, four terms, forty sections and twenty-five students, in one transaction: courses and
terms first, flushed so that the sections can be given their objects, then sections and students.
`read_rows` opens every file with `encoding="utf-8-sig"`, which reads a file the same way whether or
not it begins with a byte-order mark, for a reason the Common errors show.

### A year's enrollments, a savepoint for every row

`load_enrollments` looks up every student by email and every section by course code and term, then
adds the rows one at a time, each inside `session.begin_nested()`:


In [7]:
with Session(engine) as session:
    refused = load_enrollments(session, DATA / "enrollments.csv")
    print(session.scalar(select(func.count()).select_from(Enrollment)), "enrollments loaded")
for line, reason in refused:
    print(f"line {line}: {reason}")


228 enrollments loaded
line 230: unknown: znakamura@college.edu
line 231: UNIQUE constraint failed: enrollments.student_id, enrollments.section_id
line 232: CHECK constraint failed: ck_enrollments_status_known


228 rows loaded and three refused, each with its line in the file and the reason. Line 230 names an
email no student has, which the lookup found before the database was asked. Line 231 repeats the
first row, and line 232 has a status the table's `CHECK` does not allow; the database refused both,
and each refusal rolled back only its own savepoint. The savepoints are real on SQLite because of
`autocommit=False`, as the **Connections and Transactions** notebook showed. The one `commit()` at
the end saved every row that got in.

### The registrar's questions

`queries.py` holds the questions the registrar asks most, each answered by the database in one
statement: a student's transcript, and the **Joins and Aggregates** notebook's standing by program:


In [8]:
%%writefile scratch/queries.py
"""The registrar's questions, each answered by the database in one statement."""
from sqlalchemy import case, func, select

from models import Course, Enrollment, Section, Student, Term

POINTS = case(                                                      # grade points in tenths, so that every sum is whole
    {"A": 40, "A-": 37, "B+": 33, "B": 30, "B-": 27, "C+": 23, "C": 20, "C-": 17, "D": 10, "F": 0},
    value=Enrollment.grade,
)


def transcript(session, email):
    """A student's courses in the order they were taken: term, course code, grade."""
    return session.execute(
        select(Term.name, Course.code, Enrollment.grade)
        .join(Enrollment.student).join(Enrollment.section).join(Section.term).join(Section.course)
        .where(Student.email == email)
        .order_by(Term.starts_on, Course.code)
    ).all()


def standing(session, term):
    """For every program: the students with grades in the term, those on the dean's list, and those below 2.0."""
    per_student = (
        select(Enrollment.student_id,
               func.sum(POINTS * Course.credits).label("quality"),
               func.sum(Course.credits).label("credits"))
        .join(Enrollment.section).join(Section.course).join(Section.term)
        .where(Term.name == term, Enrollment.grade.is_not(None))
        .group_by(Enrollment.student_id)
        .subquery()
    )
    report = (
        select(Student.program, func.count(),
               func.sum(case((per_student.c.quality >= 30 * per_student.c.credits, 1), else_=0)),
               func.sum(case((per_student.c.quality < 20 * per_student.c.credits, 1), else_=0)))
        .join(per_student, per_student.c.student_id == Student.id)
        .group_by(Student.program)
        .order_by(Student.program)
    )
    return session.execute(report).all()


Writing scratch/queries.py


In [9]:
from queries import standing, transcript

with Session(engine) as session:
    for term, code, grade in transcript(session, "cmartin@college.edu"):
        print(f"{term:<12} {code:<8} {grade or 'in progress'}")
    for program, students, deans, below in standing(session, "Fall 2025"):
        print(f"{program:<17} {students} students, {deans} on the dean's list, {below} below 2.0")


Fall 2025    CHE-110  C+
Fall 2025    CSC-201  F
Fall 2025    PSY-101  B+
Spring 2026  ENG-105  in progress
Spring 2026  MAT-120  in progress
Spring 2026  STA-200  in progress
Biology           5 students, 2 on the dean's list, 0 below 2.0
Computer Science  5 students, 2 on the dean's list, 0 below 2.0
History           5 students, 0 on the dean's list, 3 below 2.0
Mathematics       5 students, 0 on the dean's list, 3 below 2.0
Psychology        5 students, 0 on the dean's list, 3 below 2.0


Chloe Martin's six courses in the order taken, the Spring 2026 ones still in progress, and Fall
2025's standing, which matches the **Joins and Aggregates** notebook's for the same term: the
loaded data is the college's, row for row.

### Tests that build their database from the migrations

The fixtures of **Testing a Data Layer**, with one change: the engine fixture builds its database by
running the project's migrations, with Alembic's `command.upgrade`, on a new file of the run's own,
and loads the reference data into it:


In [10]:
%%writefile scratch/conftest.py
from pathlib import Path

import pytest
from alembic import command
from alembic.config import Config
from sqlalchemy.orm import Session

from database import make_engine
from load import load_reference

HERE = Path(__file__).parent


@pytest.fixture(scope="session")
def migrations(tmp_path_factory):
    """The project's Alembic settings, pointed at a new database of the run's own."""
    config = Config(HERE / "alembic.ini")
    config.set_main_option("sqlalchemy.url", f"sqlite:///{tmp_path_factory.mktemp('db') / 'test.db'}")
    return config


@pytest.fixture(scope="session")
def engine(migrations):
    """The test database, built by the migrations, with the exports' courses, terms, sections and students."""
    command.upgrade(migrations, "head")
    engine = make_engine(migrations.get_main_option("sqlalchemy.url"))
    with Session(engine) as session:
        load_reference(session, HERE / "data")
    yield engine
    engine.dispose()


@pytest.fixture
def session(engine):
    """A session for one test, inside a transaction that is rolled back when the test ends."""
    with engine.connect() as connection:
        transaction = connection.begin()
        with Session(bind=connection, join_transaction_mode="create_savepoint") as session:
            yield session
        transaction.rollback()


Writing scratch/conftest.py


Four tests. The first runs `command.check`, which autogenerates in memory and raises if the models
have changed in a way no revision covers yet, so a model changed without a migration fails the
suite:


In [11]:
%%writefile scratch/test_data_layer.py
from pathlib import Path

from alembic import command

from load import load_enrollments
from queries import standing, transcript

DATA = Path(__file__).parent / "data"


def test_the_migrations_match_the_models(migrations, engine):
    command.check(migrations)                                       # raises if autogenerate would find a change


def test_three_rows_are_refused(session):
    refused = load_enrollments(session, DATA / "enrollments.csv")
    assert [line for line, reason in refused] == [230, 231, 232]


def test_every_student_had_grades_in_fall_2025(session):
    load_enrollments(session, DATA / "enrollments.csv")
    students = sum(row[1] for row in standing(session, "Fall 2025"))
    assert students == 25


def test_a_transcript_runs_in_term_order(session):
    load_enrollments(session, DATA / "enrollments.csv")
    terms = [term for term, code, grade in transcript(session, "cmartin@college.edu")]
    assert terms == ["Fall 2025"] * 3 + ["Spring 2026"] * 3


Writing scratch/test_data_layer.py


In [12]:
print(pytest_report("-q"))


....                                                                     [100%]
4 passed


Four tests passing against a database the migrations built, each loading the enrollments inside its
own rolled-back transaction, so every one starts from the reference data alone.

### Which piece does which job

| The job | The piece | What it relies on |
|---|---|---|
| describe the tables | `models.py` | Declarative Models, Relationships |
| build and change the tables | `migrations/`, from `alembic revision --autogenerate` | Migrations with Alembic |
| make an engine the same way everywhere | `database.py` | Engines and URLs, Connections and Transactions |
| bring data in from outside | `load.py`: text to Python at the edge, a savepoint for every row | The Session |
| answer questions | `queries.py`: one statement for every answer | Joins and Aggregates |
| check all of it | `conftest.py` and the tests, on a database the migrations built | Testing a Data Layer |
| run it from nothing | `build.py` | this notebook |

Everything outside the data layer, a web page, a report, a command line, calls its functions with a
session, and none of it writes SQL of its own.

### One script, from nothing, finished

The pieces of this notebook, and of the guide, in one script. `build.py` takes the name of a
database that does not exist yet, runs the migrations to make it, loads the exports, and reports:


In [13]:
%%writefile scratch/build.py
"""Build the college's database from the exports, from nothing: python build.py DATABASE"""
import sys
from pathlib import Path

from alembic import command
from alembic.config import Config
from sqlalchemy import func, select
from sqlalchemy.orm import Session

from database import make_engine
from load import load_enrollments, load_reference
from models import Enrollment, Student
from queries import standing

HERE = Path(__file__).parent
url = f"sqlite:///{HERE / sys.argv[1]}"

config = Config(HERE / "alembic.ini")
config.set_main_option("sqlalchemy.url", url)
command.upgrade(config, "head")

engine = make_engine(url)
with Session(engine) as session:
    load_reference(session, HERE / "data")
    refused = load_enrollments(session, HERE / "data" / "enrollments.csv")
    print(session.scalar(select(func.count()).select_from(Student)), "students,",
          session.scalar(select(func.count()).select_from(Enrollment)), "enrollments loaded")
    for line, reason in refused:
        print(f"refused, line {line}: {reason}")
    for program, students, deans, below in standing(session, "Fall 2025"):
        print(f"{program:<17} {students} students, {deans} on the dean's list, {below} below 2.0")
engine.dispose()


Writing scratch/build.py


In [14]:
done = subprocess.run([sys.executable, "build.py", "rebuilt.db"], cwd=SCRATCH, capture_output=True, text=True)
print(done.stdout.rstrip())
loggers = sorted({line.split("]")[0] + "]" for line in done.stderr.splitlines()})
print("exit code", done.returncode, "| what wrote to stderr:", loggers)


25 students, 228 enrollments loaded
refused, line 230: unknown: znakamura@college.edu
refused, line 231: UNIQUE constraint failed: enrollments.student_id, enrollments.section_id
refused, line 232: CHECK constraint failed: ck_enrollments_status_known
Biology           5 students, 2 on the dean's list, 0 below 2.0
Computer Science  5 students, 2 on the dean's list, 0 below 2.0
History           5 students, 0 on the dean's list, 3 below 2.0
Mathematics       5 students, 0 on the dean's list, 3 below 2.0
Psychology        5 students, 0 on the dean's list, 3 below 2.0
exit code 0 | what wrote to stderr: ['INFO  [alembic.runtime.migration]']


A new database, `rebuilt.db`, made by the migration and filled from the exports in one command: the
same 228 enrollments, the same three refusals, and the same standing as the database this notebook
built a step at a time. Alembic logs to standard error, and the last line shows only which of its
loggers wrote there, the migration runner. This script, or one like it, is what runs next month on a
newer export, or on a colleague's machine that has none of the college's data yet.

### Where each part came from

| In `build.py` | What it relies on | The section that showed it |
|---|---|---|
| `command.upgrade(config, "head")` on a new database | the schema, from a migration | The schema, from a migration |
| `make_engine(url)` | one place to make an engine | The project: models, and one place to make an engine |
| `load_reference(session, ...)` | text turned into Python at the edge | Reference data from the exports |
| `load_enrollments(session, ...)` and the refused lines | a savepoint for every row | A year's enrollments, a savepoint for every row |
| `standing(session, "Fall 2025")` | a question answered in one statement | The registrar's questions |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/20-a-complete-data-layer-solutions.ipynb).

**1.** Print the transcript of Aoife O'Brien, whose email is `aobrien@college.edu`.


In [15]:
# your code here


**2.** Write a CSV file with two new enrollments for Fall 2025, one of them for a section that does
not exist, load it with `load_enrollments`, and print what was refused.


In [16]:
# your code here


**3.** Add a query to `queries.py`, `seats_taken(session, term)`, that counts the enrollments in
every section of a term, and print the three fullest sections of Spring 2026.


In [17]:
# your code here


**4.** Add a test that `load_enrollments` refuses a row whose grade is not one of the college's
grades. (The table has no `CHECK` on grades, so the loader has to.)


In [18]:
# your code here


**5.** Give `Student.program` an index in `models.py`, with `index=True`, run the tests, and read
the failure. Then write the revision that fixes it, and run the tests again.


In [19]:
# your code here


**6.** Run `build.py` for a second new database, and compare its enrollment count with
`college.db`'s.


In [20]:
# your code here


## Common errors

### sqlalchemy.exc.OperationalError: (sqlite3.OperationalError) table courses already exists


In [21]:
edit(SCRATCH / "migrations" / "env.py", "config = context.config\n", """config = context.config
database = context.get_x_argument(as_dictionary=True).get("db")
if database:                                                        # alembic -x db=FILE works on another database
    config.set_main_option("sqlalchemy.url", f"sqlite:///{database}")
""")

from models import Base

quick = college_engine(SCRATCH / "quick.db")                       # a database someone built the quick way
Base.metadata.create_all(quick)
quick.dispose()
alembic("-x", "db=quick.db", "upgrade", "head")


$ alembic -x db=quick.db upgrade head
    INFO  [alembic.runtime.migration] Running upgrade  -> 0001, the college
    Traceback (most recent call last): ...
    sqlalchemy.exc.OperationalError: (sqlite3.OperationalError) table courses already exists


First, `env.py` learns to take a database's file from the command line, `-x db=...`, so that one
environment can migrate more than one database. Then a colleague's `quick.db`, built with
`create_all` before the migrations existed: every table, and no `alembic_version`. Alembic took it
for a database that has had no revision, ran 0001, and 0001's first `CREATE TABLE` met a table that
was already there. When the tables already match a revision, record that instead of running it:
`stamp` writes the revision into `alembic_version` and runs nothing.


In [22]:
alembic("-x", "db=quick.db", "stamp", "head")
alembic("-x", "db=quick.db", "current")
alembic("-x", "db=quick.db", "upgrade", "head")


$ alembic -x db=quick.db stamp head
    INFO  [alembic.runtime.migration] Running stamp_revision  -> 0001
$ alembic -x db=quick.db current
    0001 (head)
$ alembic -x db=quick.db upgrade head


`current` now reports 0001, and `upgrade head` has nothing to run. From here on, `quick.db` takes
every later revision like any other copy of the database.


### sqlalchemy.exc.StatementError: (builtins.TypeError) SQLite Date type only accepts Python date objects as input.


In [23]:
with open(DATA / "students.csv", newline="", encoding="utf-8") as file:
    first = next(csv.DictReader(file))
print(first)
with Session(college_engine()) as session:
    Base.metadata.create_all(session.get_bind())
    session.add(Student(name=first["name"], email=first["email"], program=first["program"], started_on=first["started_on"]))
    try:
        session.commit()
    except sqlalchemy.exc.StatementError as error:                  # its [parameters: ...] line changes order between runs
        print("StatementError:", str(error).splitlines()[0])


{'name': 'Ana Reyes', 'email': 'areyes@college.edu', 'program': 'Biology', 'started_on': '2024-08-26'}
StatementError: (builtins.TypeError) SQLite Date type only accepts Python date objects as input.


Every value from `csv.DictReader` is a string, and `started_on` arrived as `'2024-08-26'`, which
looks like a date and is not one. SQLAlchemy's `Date` type takes a `date` and refused the string
when the commit sent it. Turn the text into the column's type at the edge, as `load_reference` does:


In [24]:
with Session(college_engine()) as session:
    Base.metadata.create_all(session.get_bind())
    session.add(Student(name=first["name"], email=first["email"], program=first["program"],
                        started_on=date.fromisoformat(first["started_on"])))
    session.commit()
    print(session.scalars(select(Student)).one().started_on)


2024-08-26


### KeyError: 'code'


In [25]:
EXCEL_EXPORT = SCRATCH / "courses_from_excel.csv"                  # the same courses, saved as Excel's "CSV UTF-8"
EXCEL_EXPORT.write_text((DATA / "courses.csv").read_text(), encoding="utf-8-sig")

with open(EXCEL_EXPORT, newline="", encoding="utf-8") as file:
    row = next(csv.DictReader(file))
print(list(row))
print(row["code"])


['\ufeffcode', 'title', 'department', 'credits']


KeyError: 'code'

The header looks like `code`, and the dictionary's first key is `'﻿code'`: a byte-order mark, the
character some programs, Excel among them, write at the start of a UTF-8 file, which `utf-8` reads
as part of the first header. `utf-8-sig` reads the mark as a mark and drops it, and reads a file
without one the same as `utf-8` does, which is why `read_rows` uses it for every file:


In [26]:
from load import read_rows

print(read_rows(EXCEL_EXPORT)[0]["code"], "|", read_rows(DATA / "courses.csv")[0]["code"])


BIO-101 | BIO-101


### sqlalchemy.exc.IntegrityError: (sqlite3.IntegrityError) UNIQUE constraint failed: enrollments.student_id, enrollments.section_id


In [27]:
from load import read_rows

fresh = college_engine()                                            # a new database in memory, with the reference data
Base.metadata.create_all(fresh)
with Session(fresh) as session:
    load_reference(session, DATA)
    students = dict(session.execute(select(Student.email, Student.id)).all())
    sections = {(code, term): section_id for section_id, code, term in
                session.execute(select(Section.id, Course.code, Term.name).join(Section.course).join(Section.term))}
ROWS = [(line, row) for line, row in enumerate(read_rows(DATA / "enrollments.csv"), start=2) if row["email"] in students]


def enrollment(row):
    return Enrollment(student_id=students[row["email"]], section_id=sections[row["course"], row["term"]],
                      status=row["status"], grade=row["grade"] or None)


with Session(fresh) as session:
    session.add_all(enrollment(row) for line, row in ROWS)
    session.commit()


IntegrityError: (sqlite3.IntegrityError) UNIQUE constraint failed: enrollments.student_id, enrollments.section_id
[SQL: INSERT INTO enrollments (student_id, section_id, status, grade) VALUES (?, ?, ?, ?)]
[parameters: [(1, 2, 'completed', 'C+'), (1, 5, 'completed', 'D'), (1, 8, 'completed', 'A-'), (1, 13, 'completed', 'A'), (1, 16, 'completed', 'B'), (1, 19, 'completed', 'C'), (1, 24, 'completed', 'C+'), (1, 27, 'completed', 'D')  ... displaying 10 of 230 total bound parameter sets ...  (1, 2, 'completed', 'C+'), (1, 39, 'dropped', None)]]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

All the rows went into one flush, the database refused the repeated one, and the flush was rolled
back whole: 228 good rows lost to one bad one, and nothing loaded. Catching the error around each row
is not enough on its own, since after a failed flush the session refuses everything until it is
rolled back:


In [28]:
with Session(fresh) as session:
    flushed = 0
    for line, row in ROWS:
        try:
            session.add(enrollment(row))
            session.flush()
            flushed += 1
        except IntegrityError as error:
            print(f"line {line}: {error.orig}")
        except sqlalchemy.exc.PendingRollbackError:
            print(f"line {line}: refused, since the session is waiting for a rollback")
            break
print(flushed, "rows flushed, and none of them committed")


line 231: UNIQUE constraint failed: enrollments.student_id, enrollments.section_id
line 232: refused, since the session is waiting for a rollback
228 rows flushed, and none of them committed


The loop caught the repeated row on line 231, and line 232 was refused only because the session was
still waiting for its rollback. A `session.rollback()` there would have taken back the 228 rows
before it as well. A savepoint for every row rolls back that row alone, which is what
`load_enrollments` does:


In [29]:
with Session(fresh) as session:
    refused = load_enrollments(session, DATA / "enrollments.csv")
    print(session.scalar(select(func.count()).select_from(Enrollment)), "enrollments, refused lines:",
          [line for line, reason in refused])
fresh.dispose()


228 enrollments, refused lines: [230, 231, 232]


Last, the engine lets go of the file, and this cell removes the scratch folder, with the project, its
exports and its databases in it:


In [30]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A data layer is models, migrations, loaders, queries and tests, in modules whose functions take the
  session they work in; everything else goes through it.
- The first migration builds the schema, so the database knows its revision from the start; a
  database built with `create_all` first is `stamp`ed, not upgraded.
- A loader turns text into Python at the edge, reads CSV files with `utf-8-sig`, and loads every row
  in a savepoint of its own, reporting the line and the reason of each row refused.
- Tests build their database from the same migrations and check with `command.check` that the
  models have no change a revision does not cover.
- One script runs it all from nothing, which is how the next export, or the next machine, gets its
  database.


## What is next

That is the end of this guide. You can reach a database through an engine and its connections, read
results in the shape the code needs, describe tables with `MetaData` and with mapped classes, write
SQL as Python expressions, work through a session and its identity map, relate classes one to many
and many to many, choose how related objects load, join and aggregate, decide what a delete does to
the rows that point at it, do the same work asynchronously, change a schema with Alembic, write for
four databases from one set of models, test a data layer, and put all of it together from nothing.

The **SQLModel, Deep Dive** guide comes next. SQLModel is built on SQLAlchemy's ORM and on Pydantic,
so that one class is both a table and a model that FastAPI can check a request against.


---

&#8592; **Previous:** [Testing a Data Layer](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/19-testing-a-data-layer.ipynb)  &nbsp;·&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
